# OULAD pipeline Another approach

## 1. Setup

In [20]:
%pip install networkx matplotlib -q
%pip install pyvis -q  # interactive graphs
%pip install torch-geometric -q # for GNN later


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [21]:

# ============================================
# OULAD Cross-Course Evaluation
# Week 2 / 4 / 6 / 8
# ============================================

import sys
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    balanced_accuracy_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


## Reproducible paths

In [22]:
# Configure paths - handle both execution contexts
if Path.cwd().name == 'notebooks':
    project_root = Path.cwd().parent
else:
    project_root = Path.cwd()

# Add src to path and import centralized configuration
sys.path.insert(0, str(project_root / 'src'))

from config import (
    DATA_DIR,
    RESULTS_DIR,
    STUDENT_INFO_FILE,
    STUDENT_VLE_FILE,
    STUDENT_ASSESSMENT_FILE,
    ASSESSMENTS_FILE,
    COURSES_FILE,
    VLE_FILE,
    LABEL_MAPPING,
    PREDICTION_WINDOWS,
    MODEL_PARAMS,
    RANDOM_STATE
)

# Use imported paths directly
studentInfo_path = STUDENT_INFO_FILE
studentVle_path = STUDENT_VLE_FILE
studentAssessment_path = STUDENT_ASSESSMENT_FILE
assessments_path = ASSESSMENTS_FILE

print("✓ Configuration loaded successfully")
print(f"Project root: {project_root}")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"\nData files:")
print(f"  studentInfo: {studentInfo_path.name}")
print(f"  studentVle: {studentVle_path.name}")
print(f"  studentAssessment: {studentAssessment_path.name}")
print(f"  assessments: {assessments_path.name}")

✓ Configuration loaded successfully
Project root: /Users/olivialoza/Documents/Development/OULAD
Data directory: /Users/olivialoza/Documents/Development/OULAD/DATA/raw
Results directory: /Users/olivialoza/Documents/Development/OULAD/results

Data files:
  studentInfo: studentInfo.csv
  studentVle: studentVle.csv
  studentAssessment: studentAssessment.csv
  assessments: assessments.csv


## 3. Load preprocessed data

In [23]:
# Verify all data files exist
print("Verifying data files...")
for file_path in [studentInfo_path, studentVle_path, studentAssessment_path, assessments_path]:
    if not file_path.exists():
        print(f"✗ Missing: {file_path}")
    else:
        print(f"✓ Found: {file_path.name}")

# Build a dictionary of available dataframes (without shadowing `datasets`)
loaded_datasets = {}

if "student_info" in globals():
    loaded_datasets["student_info"] = student_info
if "student_assess" in globals():
    loaded_datasets["student_assess"] = student_assess
if "assessments" in globals():
    loaded_datasets["assessments"] = assessments
if "courses" in globals():
    loaded_datasets["courses"] = courses

for name, df_ in loaded_datasets.items():
    print(f"\n{name}")
    print(f"Shape: {df_.shape}")

    if df_.empty:
        print("WARNING: Dataset is empty")
    else:
        print("Loaded successfully")

print(f"\nData directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")


Verifying data files...
✓ Found: studentInfo.csv
✓ Found: studentVle.csv
✓ Found: studentAssessment.csv
✓ Found: assessments.csv

student_info
Shape: (32593, 13)
Loaded successfully

Data directory: /Users/olivialoza/Documents/Development/OULAD/DATA/raw
Results directory: /Users/olivialoza/Documents/Development/OULAD/results


## 4. Define risk

In [24]:
# Binary risk:
# 1 = at-risk (Fail/Withdrawn)
# 0 = success (Pass/Distinction)
# Ensure `student_info` exists
if "student_info" not in globals():
    if studentInfo_path.exists():
        student_info = pd.read_csv(studentInfo_path)
    else:
        raise FileNotFoundError(f"Missing file: {studentInfo_path}")

# Validate required column
if "final_result" not in student_info.columns:
    raise KeyError("'final_result' column not found in student_info")

# Create binary target
student_info["target"] = student_info["final_result"].apply(
    lambda x: 1 if x in ["Fail", "Withdrawn"] else 0
)

# Keep `df` available for downstream cells if not already defined
if "df" not in globals():
    df = student_info.copy()
counts = student_info['target'].value_counts()
counts.index = counts.index.map({1: 'At-risk (1)', 0: 'Low-risk (0)'})

counts_df = counts.reset_index()
counts_df.columns = ["Category", "Count"]

# Add percentage
counts_df["Percentage (%)"] = (counts_df["Count"] / counts_df["Count"].sum() * 100).round(2)

# To make sure the target is not missing in the main dataframe, we add it if it's not present
if "target" not in df.columns and "final_result" in df.columns:
    df["target"] = df["final_result"].apply(
        lambda x: 1 if x in ["Fail", "Withdrawn"] else 0
    )

display(counts_df)


,Category,Count,Percentage (%)
0,At-risk (1),17208,52.8
1,Low-risk (0),15385,47.2


## 5. Define prediction windows

In [25]:

# Week filtering
weeks = [2, 4, 6, 8]

def filter_week(df, week):
    return df[df["week"] <= week].copy()


## 6. Feature groups

In [26]:

demographic_cols = [
    "age_band",
    "num_of_prev_attempts",
    "studied_credits"
]

vle_cols = [
    "total_clicks",
    "unique_resources",
    "activity_days"
]

assessment_cols = [
    "num_assessments",
    "mean_score",
    "submission_rate"
]

feature_groups = {
    "demographics": demographic_cols,
    "vle": vle_cols,
    "assessment": assessment_cols,
    "combined": demographic_cols + vle_cols + assessment_cols
}


## 7. Define models

In [27]:

models = {
    "logreg": LogisticRegression(max_iter=10000),
    "rf": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE
    ),
    "xgb": xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        use_label_encoder=False,
        eval_metric='logloss'
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        use_label_encoder=False,
        eval_metric='logloss'
    ,
        validate_parameters=(globals().__setitem__("lgb", __import__("lightgbm")) or True)),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=RANDOM_STATE
     )  

}


## 8. Metrics functions

In [28]:

def compute_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    
    return {
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred)
    }


## 9 Random  (Basaeline)

In [29]:

def run_random_split(df, features, model, week):
    
    df_w = filter_week(df, week)
    
    X = df_w[features]
    y = df_w["at_risk"]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:,1]
    
    metrics = compute_metrics(y_test, y_prob)
    metrics["week"] = week
    
    return metrics


## 10. LCPO Split (Core contribution)

In [30]:

def run_lcpo(df, features, model, week):
    
    df_w = filter_week(df, week)
    
    X = df_w[features]
    y = df_w["at_risk"]
    
    groups = df_w["code_module"] + "_" + df_w["code_presentation"]
    
    gkf = GroupKFold(n_splits=len(groups.unique()))
    
    results = []
    
    for train_idx, test_idx in gkf.split(X, y, groups):
        
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:,1]

        metrics = compute_metrics(y_test, y_prob)
        
        metrics["week"] = week
        metrics["test_course"] = groups.iloc[test_idx].iloc[0]

        results.append(metrics)

    return pd.DataFrame(results)


## 11. Main experiment loop

Feature preparation with string fix

In [31]:

def clean_feature_names(df):
    df.columns = (
        df.columns
        .str.replace(r"[^\w]", "_", regex=True)   # keep only letters/numbers/_
        .str.replace(r"_+", "_", regex=True)      # remove duplicate underscores
        .str.strip("_")                           # trim edges
    )
    return df


def prepare_features(df, feature_cols):
    X = df[feature_cols].copy()

    # Detect categorical columns
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

    # One-hot encoding
    if len(cat_cols) > 0:
        X = pd.get_dummies(X, columns=cat_cols, drop_first=False)

    # Clean feature names for XGBoost compatibility
    X = clean_feature_names(X)

    # Ensure numeric
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    return X


Random split

In [32]:

def run_random_split(df, features, model, week):
    
    df_w = filter_week(df, week)

    X = prepare_features(df_w, features)
    y = df_w["at_risk"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        stratify=y,
        test_size=0.2,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = compute_metrics(y_test, y_prob)
    metrics["week"] = week

    return metrics


LCPO Split (Cross-course) (IMplementation for this part)

In [33]:

def run_lcpo(df, features, model, week):
    
    df_w = filter_week(df, week)

    groups = df_w["code_module"] + "_" + df_w["code_presentation"]

    X = prepare_features(df_w, features)
    y = df_w["at_risk"]

    gkf = GroupKFold(n_splits=len(groups.unique()))

    results = []

    for train_idx, test_idx in gkf.split(X, y, groups):

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]

        metrics = compute_metrics(y_test, y_prob)
        metrics["week"] = week
        metrics["test_course"] = groups.iloc[test_idx].iloc[0]

        results.append(metrics)

    return pd.DataFrame(results)


Main Loop 

In [34]:

overall_results = []
course_results = []

for week in weeks:
    
    print(f"\n==============================")
    print(f"RUNNING WEEK {week}")
    print(f"==============================")
    
    for group_name, features in feature_groups.items():
        
        required_cols = set(features + ["at_risk", "code_module", "code_presentation", "week"])
        missing_cols = [c for c in required_cols if c not in df.columns]
        
        if missing_cols:
            print(f"Skipping group={group_name} (missing {missing_cols})")
            continue
        
        for model_name, model in models.items():
            
            print(f"▶ week={week} | group={group_name} | model={model_name}")
            
            try:
                # RANDOM SPLIT
                res = run_random_split(df, features, model, week)
                res["split"] = "random"
                res["model"] = model_name
                res["feature_group"] = group_name
                overall_results.append(res)

                # LCPO SPLIT
                lcpo_df = run_lcpo(df, features, model, week)
                lcpo_df["split"] = "lcpo"
                lcpo_df["model"] = model_name
                lcpo_df["feature_group"] = group_name
                course_results.append(lcpo_df)

            except Exception as e:
                print("ERROR:")
                print(f"week={week}, group={group_name}, model={model_name}")
                print(str(e))
                continue



RUNNING WEEK 2
Skipping group=demographics (missing ['at_risk', 'week'])
Skipping group=vle (missing ['total_clicks', 'at_risk', 'week', 'unique_resources', 'activity_days'])
Skipping group=assessment (missing ['submission_rate', 'mean_score', 'at_risk', 'num_assessments', 'week'])
Skipping group=combined (missing ['submission_rate', 'total_clicks', 'mean_score', 'num_assessments', 'at_risk', 'week', 'unique_resources', 'activity_days'])

RUNNING WEEK 4
Skipping group=demographics (missing ['at_risk', 'week'])
Skipping group=vle (missing ['total_clicks', 'at_risk', 'week', 'unique_resources', 'activity_days'])
Skipping group=assessment (missing ['submission_rate', 'mean_score', 'at_risk', 'num_assessments', 'week'])
Skipping group=combined (missing ['submission_rate', 'total_clicks', 'mean_score', 'num_assessments', 'at_risk', 'week', 'unique_resources', 'activity_days'])

RUNNING WEEK 6
Skipping group=demographics (missing ['at_risk', 'week'])
Skipping group=vle (missing ['total_clic

## 12.Save results

## 13. Generalization Gap plot

In [36]:
# Load existing baseline results
baseline_path = RESULTS_DIR / "baseline" / "baseline_results_detailed.csv"
results_df = pd.read_csv(baseline_path)

# Rename columns to match expected format
results_df = results_df.rename(columns={
    'Week': 'week',           # Capital W → lowercase
    'Model': 'model',         # Capital M → lowercase  
    'Features': 'feature_group',  # Different name
    'AUROC_mean': 'AUROC'     # Remove _mean suffix
})

# Add split column
results_df["split"] = "random"

print(f"✓ Loaded {len(results_df)} baseline results")

# Now cell 17 works:
random_df = (
    results_df[results_df["split"] == "random"]
    .groupby(["week", "model", "feature_group"], as_index=False)["AUROC"]
    .mean()
    .rename(columns={"AUROC": "random"})
)

✓ Loaded 68 baseline results


## 14. Course level Heat map

In [38]:

pivot = course_df.pivot_table(
    index="test_course",
    columns="week",
    values="AUROC"
)

plt.figure(figsize=(12,6))
sns.heatmap(pivot, cmap="coolwarm")
plt.title("Course-Level Generalization (AUROC)")
plt.show()


KeyError: 'test_course'

#  Graph construction and implementation 

In [ ]:

# Ensure GRAPH_DIR exists (use existing OUTPUT_DIR from the notebook)
if "GRAPH_DIR" not in globals():
    GRAPH_DIR = OUTPUT_DIR / "graph"

GRAPH_DIR.mkdir(parents=True, exist_ok=True)

student_nodes = df[[
    "id_student",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits"
]].drop_duplicates()

student_nodes.to_csv(GRAPH_DIR / "nodes_student.csv", index=False)


### Course nodes

In [ ]:

course_nodes = df[[
    "code_module",
    "code_presentation"
]].drop_duplicates()

course_nodes["course_id"] = (
    course_nodes["code_module"] + "_" + course_nodes["code_presentation"]
)

course_nodes.to_csv(GRAPH_DIR / "nodes_course.csv", index=False)


### Student -> Course edge

In [ ]:

edges_enrollment = df[[
    "id_student",
    "code_module",
    "code_presentation"
]].drop_duplicates()

edges_enrollment["course_id"] = (
    edges_enrollment["code_module"] + "_" + edges_enrollment["code_presentation"]
)

edges_enrollment.to_csv(GRAPH_DIR / "edges_enrollment.csv", index=False)


### Student -> vle edge 

In [ ]:
if "total_clicks" in df.columns:
    edges_vle = df[["id_student", "total_clicks", "week"]].copy()
else:
    if "student_vle" not in globals():
        if not studentVle_path.exists():
            raise FileNotFoundError(f"Missing file: {studentVle_path}")
        student_vle = pd.read_csv(studentVle_path)

    if "sum_click" not in student_vle.columns:
        raise KeyError("Neither 'total_clicks' in df nor 'sum_click' in student_vle.")

    tmp_vle = student_vle.copy()

    if "week" not in tmp_vle.columns:
        if "date" in tmp_vle.columns:
            tmp_vle["week"] = (tmp_vle["date"].clip(lower=0) // 7 + 1).astype(int)
        else:
            tmp_vle["week"] = 1

    edges_vle = (
        tmp_vle.groupby(["id_student", "week"], as_index=False)["sum_click"]
        .sum()
        .rename(columns={"sum_click": "total_clicks"})
    )

edges_vle.rename(columns={"total_clicks": "weight"}, inplace=True)

edges_vle.to_csv(GRAPH_DIR / "edges_student_vle.csv", index=False)


### Student -> Assessment edge

In [ ]:
# Build Student -> Assessment edge robustly
if {"mean_score", "submission_rate", "week"}.issubset(df.columns):
    edges_assessment = df[
        ["id_student", "mean_score", "submission_rate", "week"]
    ].copy()
else:
    # Load studentAssessment if not already in memory
    if "student_assess" not in globals():
        if not studentAssessment_path.exists():
            raise FileNotFoundError(f"Missing file: {studentAssessment_path}")
        student_assess = pd.read_csv(studentAssessment_path)

    tmp_assess = student_assess.copy()

    # Build week
    if "week" not in tmp_assess.columns:
        if "date_submitted" in tmp_assess.columns:
            tmp_assess["week"] = (tmp_assess["date_submitted"].clip(lower=0) // 7 + 1).astype(int)
        elif "date" in tmp_assess.columns:
            tmp_assess["week"] = (tmp_assess["date"].clip(lower=0) // 7 + 1).astype(int)
        else:
            tmp_assess["week"] = 1

    # Ensure score exists
    if "score" not in tmp_assess.columns:
        raise KeyError("Neither df has assessment features nor student_assess has 'score'.")

    # Compute mean_score + submission_rate per student-week
    tmp_assess["submitted"] = tmp_assess["score"].notna().astype(int)
    edges_assessment = (
        tmp_assess.groupby(["id_student", "week"], as_index=False)
        .agg(
            mean_score=("score", "mean"),
            submission_rate=("submitted", "mean")
        )
        .fillna({"mean_score": 0.0, "submission_rate": 0.0})
    )

edges_assessment.to_csv(GRAPH_DIR / "edges_student_assessment.csv", index=False)


## Graph

Imports

In [ ]:

import networkx as nx
import matplotlib.pyplot as plt

import pandas as pd


Load data

In [ ]:
candidate_paths = [
    DATA_DIR / "oulad_student_course_week.csv",
    PROCESSED_DIR / "oulad_student_course_week.csv",
    RAW_DIR / "oulad_student_course_week.csv",
]

existing_path = next((p for p in candidate_paths if p.exists()), None)

if existing_path is not None:
    df = pd.read_csv(existing_path)
    print(f"Loaded: {existing_path}")
elif "df" in globals():
    print("File not found in expected folders. Using existing `df` already in memory.")
else:
    searched = "\n".join(str(p) for p in candidate_paths)
    raise FileNotFoundError(
        "Could not find 'oulad_student_course_week.csv'. Searched:\n" + searched
    )

df.head()


Smaple set

In [ ]:

# Small sample for visualization
df_sample = df.sample(n=200, random_state=42)
G = nx.Graph()

Student -> Course graph

In [ ]:

G_sc = nx.Graph()

for _, row in df_sample.iterrows():
    
    student = f"student_{row['id_student']}"
    course = f"course_{row['code_module']}_{row['code_presentation']}"
    
    G_sc.add_node(student, type="student")
    G_sc.add_node(course, type="course")
    
    G_sc.add_edge(student, course)
#plt.figure(figsize=(12, 36)) 
#nx.draw(G_sc, with_labels=True, node_size=500, node_color="lightblue")
#plt.title("Student-Course Enrollment Graph (Sample)")
#plt.show()

plt.figure(figsize=(12,8))

pos = nx.spring_layout(G_sc, seed=42)

# Node colors
colors = [
    "blue" if G_sc.nodes[n]["type"] == "student" else "red"
    for n in G_sc.nodes()
]

nx.draw(G_sc, pos,
        node_color=colors,
        with_labels=False,
        node_size=50,
        edge_color="gray")

plt.title("Student → Course Graph")
plt.show()


Student -> VLE graph

In [ ]:


G_sv = nx.Graph()

for _, row in df_sample.iterrows():
    
    student = f"student_{row['id_student']}"
    vle_node = f"vle_week_{row['week']}"
    
    weight = row.get("total_clicks", 0)
    
    G_sv.add_node(student, type="student")
    G_sv.add_node(vle_node, type="vle")
    
    G_sv.add_edge(student, vle_node, weight=weight)


plt.figure(figsize=(10,8))

pos = nx.spring_layout(G_sv, seed=42)

colors = [
    "blue" if G_sv.nodes[n]["type"] == "student" else "green"
    for n in G_sv.nodes()
]

nx.draw(G_sv, pos,
        node_color=colors,
        with_labels=False,
        node_size=50,
        edge_color="gray")

plt.title("Student → VLE Interaction Graph")
plt.show()


Student -> Assessment graph

In [ ]:

G_sa = nx.Graph()

for _, row in df_sample.iterrows():
    
    student = f"student_{row['id_student']}"
    assessment = f"assessment_week_{row['week']}"
    
    score = row.get("mean_score", 0)
    
    G_sa.add_node(student, type="student")
    G_sa.add_node(assessment, type="assessment")
    
    G_sa.add_edge(student, assessment, weight=score)


plt.figure(figsize=(10,8))

pos = nx.spring_layout(G_sa, seed=42)

colors = [
    "blue" if G_sa.nodes[n]["type"] == "student" else "orange"
    for n in G_sa.nodes()
]

nx.draw(G_sa, pos,
        node_color=colors,
        with_labels=False,
        node_size=50,
        edge_color="gray")

plt.title("Student → Assessment Graph")
plt.show()



Combined graph

In [ ]:

G_all = nx.Graph()

for _, row in df_sample.iterrows():
    
    student = f"student_{row['id_student']}"
    course = f"course_{row['code_module']}_{row['code_presentation']}"
    vle = f"vle_week_{row['week']}"
    assessment = f"assessment_week_{row['week']}"
    
    G_all.add_node(student, type="student")
    G_all.add_node(course, type="course")
    G_all.add_node(vle, type="vle")
    G_all.add_node(assessment, type="assessment")

    G_all.add_edge(student, course)
    G_all.add_edge(student, vle)
    G_all.add_edge(student, assessment)



plt.figure(figsize=(12,10))

pos = nx.spring_layout(G_all, seed=42)

color_map = {
    "student": "blue",
    "course": "red",
    "vle": "green",
    "assessment": "orange"
}

colors = [color_map[G_all.nodes[n]["type"]] for n in G_all.nodes()]

nx.draw(G_all, pos,
        node_color=colors,
        with_labels=False,
        node_size=50,
        edge_color="gray")

plt.title("Heterogeneous Graph (Student + Course + VLE + Assessment)")
plt.show()
